<img src="https://raw.githubusercontent.com/pasdptt/public_teaching_tools4DS/main/assets/brand/logo-160.png" width="80" align="right" alt="Course logo">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasdptt/public_teaching_tools4DS/blob/main/modules/m07-networks/lab/lab7b_centrality_communities.ipynb)

# Lab 7B — Centrality, communities and link prediction

In [ ]:
# Setup: run this cell first (Google Colab or a local Jupyter kernel).
COURSE_REF = "main"  # version of the course repository to install the helper package from
try:
    import tsds
except ImportError:  # first run in a fresh Colab runtime: install the course helper package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tsds @ git+https://github.com/pasdptt/public_teaching_tools4DS@{COURSE_REF}#subdirectory=src"], check=True)
    import tsds

tsds.env.check()          # prints versions; warns if a package is outside the tested range
SEED = tsds.seed_everything()
rng = tsds.rng()

import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics import roc_auc_score, adjusted_rand_score

**Course:** 040563107 Tools for Statistical Data Science · **Module m07** (week 2 of 2) · **Time:** 3 hours · **Graded lab G4** (3⅓% of the course)

**After this lab you can:**
* compute PageRank as the stationary vector of a random walk by power iteration, and compare centrality measures (objective m07-O4);
* test whether a network statistic is surprising against a **configuration-model null** that keeps every degree (O4, O5);
* detect communities by modularity and judge them against the null and a known split (O5);
* evaluate link prediction with a correct **edge split**, and show how computing features on the full graph leaks the answer (O6, failure analysis).

**Two steps:** in the lab session we **run through** the ideas together. Then, as homework, you **do this lab on your own**
from a fresh copy and upload the `.ipynb` to the G4 assignment in Microsoft Teams. **That homework is what is marked.**

**How it is marked** ([lab rubric](https://github.com/pasdptt/public_teaching_tools4DS/blob/main/course/rubrics.md#lab-rubric-graded-labs)):
your notebook is re-run from top to bottom in a fresh session and checked with hidden tests (correctness 3 points,
reproducibility 1 point); staff mark your written answers (failure analysis 2, interpretation 2).

**How to work:**
1. Run the setup cell first.
2. Complete every cell marked **Task**. Replace `raise NotImplementedError()` with your code, then run the **Self-check** cell below it; it prints `OK` when your answer passes. Do not edit given cells or self-check cells.
3. Write the answers marked **Q** after *Your answer:* (at most 100 words each).
4. Before submitting, use *Runtime → Restart session and run all*. The notebook must run top to bottom without errors.
5. Fill in the **AI-use log** at the end.

## Part 1 — Who is central? PageRank as an eigenproblem (about 35 minutes)

Zachary's karate club: 34 members of a university club and the 78 friendships observed outside the club. During the study the
club split in two after a conflict between the instructor (node 0) and the administrator (node 33); the node attribute `club`
records which side each member joined. We use the friendships without weights.

**PageRank.** A random surfer follows a random edge out of the current node with probability $\alpha$ and jumps to a uniformly
random node with probability $1 - \alpha$; a node with no out-edges (a *dangling* node) jumps uniformly. With $P_{ij}$ the
probability of moving from $i$ to $j$, the PageRank vector $\pi$ is the stationary distribution

$$\pi^\top = \pi^\top \left( \alpha P + (1 - \alpha) \tfrac{1}{n} \mathbf{1}\mathbf{1}^\top \right),$$

the leading left eigenvector (eigenvalue 1) of the "Google matrix". **Power iteration:** start from $\pi = \mathbf{1}/n$ and
repeat $\pi \leftarrow$ (one step of the walk) until the change is tiny. Compare PCA, where the leading eigenvector of a
covariance matrix was found the same way.

In [ ]:
K = nx.karate_club_graph()
Gk = nx.Graph(K.edges())          # unweighted friendships
club = np.array([K.nodes[v]["club"] == "Officer" for v in sorted(Gk)], dtype=int)
print(Gk.number_of_nodes(), "members,", Gk.number_of_edges(), "friendships; sides:", np.bincount(club))

In [ ]:
# Task T1
# pagerank_power(G, alpha=0.85, tol=1e-12, max_iter=1000): PageRank of a directed or undirected networkx graph (edges unweighted;
# an undirected edge counts in both directions) by power iteration. Dangling nodes jump uniformly.
# Return a dict node -> score; the scores sum to 1. Stop when the L1 change between iterations is below tol.
# Then pr: pagerank_power(Gk), and top3: the three most central nodes by pr, most central first (a list).
def pagerank_power(G, alpha=0.85, tol=1e-12, max_iter=1000):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
cent = pd.DataFrame({"degree": dict(Gk.degree()), "betweenness": nx.betweenness_centrality(Gk),
                     "eigenvector": nx.eigenvector_centrality_numpy(Gk), "pagerank": pr})
print("top 3 by PageRank:", top3)
cent.loc[[0, 33, 32, 2]].round(3)

In [ ]:
# Self-check for task T1: run it; it should print OK.
ref = nx.pagerank(Gk, alpha=0.85, tol=1e-12)
assert abs(sum(pr.values()) - 1) < 1e-9
assert max(abs(pr[v] - ref[v]) for v in Gk) < 1e-6, "compare with nx.pagerank"
assert top3 == [33, 0, 32]
print("OK")

**Q1.** **Interpretation.** Compare the centrality measures for nodes 0, 33, 32 and 2. Which members are central by every measure, and
where do the measures disagree? Relate what PageRank computes to an eigenvector, as in PCA.

*Your answer:* 

## Part 2 — Is the clustering surprising? A configuration-model null (about 35 minutes)

The karate club has many triangles (friends of friends are friends). But members with many friends create triangles almost
automatically. A fair comparison keeps every member's degree and randomises **who** is connected to whom: the
**configuration model**. We sample it by repeated **double-edge swaps** ($a\!-\!b,\ c\!-\!d \to a\!-\!d,\ c\!-\!b$), which keep
all degrees and keep the graph simple.

The empirical $p$-value of an observed statistic $T_{obs}$ against $R$ null samples is $\big(1 + \#\{T_r \ge T_{obs}\}\big)/(R + 1)$.

In [ ]:
# Task T2
# (a) null_samples(G, stat, n_samples, seed): for r = 0, 1, ..., n_samples - 1: copy G, apply
#     nx.double_edge_swap(H, nswap=10 * m, max_tries=1000 * m, seed=seed + r) with m = number of edges, and record stat(H).
#     Return a numpy array of the n_samples values.
# (b) trans_obs = nx.transitivity(Gk); trans_null = null_samples(Gk, nx.transitivity, 200, SEED)
#     trans_z = (trans_obs - mean of trans_null) / sd of trans_null (np.std); trans_p = the empirical p-value above
def null_samples(G, stat, n_samples, seed):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
print(f"transitivity {trans_obs:.4f} | null mean {trans_null.mean():.4f} (sd {trans_null.std():.4f}) | z = {trans_z:.2f}, p = {trans_p:.3f}")

In [ ]:
# Self-check for task T2: run it; it should print OK.
deg = sorted(d for _, d in Gk.degree())
same = null_samples(Gk, lambda H: sorted(d for _, d in H.degree()) == deg and nx.number_of_selfloops(H) == 0, 5, 1)
assert same.all(), "every null graph must keep all degrees and stay simple"
assert len(trans_null) == 200 and abs(trans_obs - 0.2557) < 1e-4
assert abs(trans_null.mean() - 0.2218) < 0.01 and 0 < trans_p < 1
print("OK")

## Part 3 — Communities and modularity (about 30 minutes)

**Modularity** $Q$ compares the edges inside communities with the number expected if edges were placed at random **given the
degrees**: $Q = \frac{1}{2m}\sum_{ij}\big(A_{ij} - \frac{k_i k_j}{2m}\big)\,[c_i = c_j]$. The Louvain method searches for a partition
with high $Q$. But a search finds *some* structure in any graph, so we compare the best $Q$ found with the best $Q$ found on null
graphs, and the communities with the known split.

In [ ]:
# Task T3
# (a) comm = nx.community.louvain_communities(Gk, seed=SEED); Q_obs = nx.community.modularity(Gk, comm)
# (b) Q_null: null_samples(Gk, stat, 100, SEED), where stat(H) is the modularity of louvain_communities(H, seed=SEED) on H
#     Q_p: the empirical p-value of Q_obs against Q_null
# (c) labels: a numpy array, labels[v] = index of the community containing node v (nodes 0-33)
#     ari_club = adjusted_rand_score(club, labels)
# YOUR CODE HERE
raise NotImplementedError()
print(f"{len(comm)} communities, Q = {Q_obs:.3f} | null Q: mean {Q_null.mean():.3f}, max {Q_null.max():.3f} | p = {Q_p:.3f} | ARI with the split {ari_club:.3f}")

In [ ]:
# Self-check for task T3: run it; it should print OK.
assert abs(Q_obs - 0.4188) < 0.01 and len(Q_null) == 100
assert Q_null.mean() > 0.2, "Louvain finds positive modularity even in null graphs"
assert Q_obs > Q_null.max() and abs(Q_p - 1 / 101) < 1e-12
assert labels.shape == (34,) and 0.3 < ari_club < 0.5
print("OK")

**Q2.** **Interpretation.** Your clustering test (Part 2) and your modularity test (Part 3) give different conclusions. State both with
numbers, and explain what the configuration-model null keeps fixed and why that matters here. How well do the communities match
the actual split?

*Your answer:* 

## Part 4 — Link prediction with a correct edge split (about 40 minutes)

Link prediction asks which **missing** edges are likely to exist. To evaluate it honestly we hide some edges, compute every score
on the graph **without** them, and ask whether the hidden edges score higher than pairs that are truly not connected (AUC).

We use a synthetic network with known structure, a **stochastic block model**: 120 nodes in 3 groups of 40, edge probability
0.25 within a group and 0.02 between groups. Two scores for a pair $(u, v)$, both computed from a graph `g`:

* **Adamic–Adar:** $\sum_{w \in N(u) \cap N(v)} 1 / \log d(w)$, common neighbours weighted by rarity;
* **negative distance:** $-$(length of the shortest path from $u$ to $v$).

In [ ]:
S = nx.stochastic_block_model([40, 40, 40], [[0.25, 0.02, 0.02], [0.02, 0.25, 0.02], [0.02, 0.02, 0.25]], seed=SEED)
print(S.number_of_nodes(), "nodes,", S.number_of_edges(), "edges")


def pair_scores(g, pairs):
    aa = np.array([s for _, _, s in nx.adamic_adar_index(g, pairs)])
    dist = []
    for u, v in pairs:
        try:
            dist.append(-nx.shortest_path_length(g, u, v))
        except nx.NetworkXNoPath:
            dist.append(-g.number_of_nodes())
    return {"adamic_adar": aa, "neg_distance": np.array(dist, dtype=float)}

In [ ]:
# Task T4
# split_edges(G, frac, seed): rng = np.random.default_rng(seed).
#   test_pos: the first int(frac * m) edges of rng.permutation of the edge list (m edges), as (u, v) tuples
#   G_train:  a copy of G with the test_pos edges removed (same nodes)
#   test_neg: the same number of pairs drawn without replacement from list(nx.non_edges(G)) with rng.choice
#   Return G_train, pairs = test_pos + test_neg, y = 1 for test_pos and 0 for test_neg (numpy array)
# Then: G_train, pairs, y = split_edges(S, 0.2, SEED)
#       auc_train: dict score name -> roc_auc_score(y, scores) with the scores of pair_scores(G_train, pairs)
def split_edges(G, frac, seed):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
print(f"{int(y.sum())} hidden edges, {int((1 - y).sum())} non-edges | AUC on the training graph:",
      {k: round(v, 3) for k, v in auc_train.items()})

In [ ]:
# Self-check for task T4: run it; it should print OK.
pos = [p for p, t in zip(pairs, y) if t == 1]
neg = [p for p, t in zip(pairs, y) if t == 0]
assert len(pos) == len(neg) == int(0.2 * S.number_of_edges())
assert not any(G_train.has_edge(u, v) for u, v in pos), "hidden edges must be removed from the training graph"
assert not any(S.has_edge(u, v) for u, v in neg), "negatives must be non-edges of the full graph"
assert set(G_train) == set(S) and G_train.number_of_edges() == S.number_of_edges() - len(pos)
assert 0.55 < auc_train["adamic_adar"] < 0.75 and 0.55 < auc_train["neg_distance"] < 0.75
print("OK")

**Q3.** **Interpretation.** What do your two AUCs mean for this network? Explain in terms of the three groups why common-neighbour scores
can rank hidden edges above non-edges, and why they do so only moderately here.

*Your answer:* 

## Part 5 — Failure analysis: scores computed on the full graph (about 20 minutes)

A common mistake is to compute the link-prediction features on the **full** graph, which still contains the hidden edges.

In [ ]:
# Task T5
# auc_full: the same dict as auc_train, but with the scores of pair_scores(S, pairs) (the full graph)
# YOUR CODE HERE
raise NotImplementedError()
print(pd.DataFrame({"training graph": auc_train, "full graph (leak)": auc_full}).round(3))

In [ ]:
# Self-check for task T5: run it; it should print OK.
assert auc_full["neg_distance"] == 1.0, "on the full graph every hidden edge has distance 1"
assert auc_full["adamic_adar"] > auc_train["adamic_adar"]
print("OK")

**Q4.** **Failure analysis.** Explain each change between the training-graph and full-graph AUCs. Why is one of them exactly 1? Name one
other way graph data can leak in link prediction, and how a correct protocol avoids it.

*Your answer:* 

## AI-use log (required; see the course AI policy)

| Tool | What you asked or used it for | What you accepted or changed | How you verified it |
|---|---|---|---|
| *e.g. Colab AI* | *how double_edge_swap keeps the degrees* | *used nswap = 10 m as the lab asks* | *T2 self-check: degrees unchanged* |
| | | | |

If you used no AI tool, write "None".

**Cite the data** when you use them: Zachary, W. W. (1977). An information flow model for conflict and fission in small groups. *Journal of Anthropological Research*, 33(4), 452–473 (the karate club network as bundled with networkx). The block-model network is synthetic.